In [1]:
from pyiron import Project
import math

In [2]:
# Create Project
project = Project(path='./Al_Defect')

In [3]:
# Create Structure Bulk
originalStructure = project.create.structure.bulk('FeAl', crystalstructure='cesiumchloride', a=2.9)
originalStructure = originalStructure.repeat([2, 2, 2])

originalStructure.plot3d()



NGLWidget()

In [4]:
# Create Vacancy Aluminium
vacancyAl = originalStructure.copy()
del vacancyAl[1]
vacancyAl.plot3d()


NGLWidget()

In [7]:
# Relax Structure (Sphinx)
relax_job = project.create.job.Sphinx('relax_defect', delete_existing_job=True)
relax_job.structure = vacancyAl
relax_job.calc_minimize()
relax_job.run() # 16 min
relaxedStructure = relax_job.get_structure()

The job relax_defect was saved and received the ID: 226


2026-06-05 09:27:33,179 - pyiron_log - WARNING - Could not access indices, returning None!
2026-06-05 09:27:33,483 - pyiron_log - WARNING - Could not access indices, returning None!


In [17]:
# Run GPAW (Original Structure)
calc_energy_og_job = project.create.job.Gpaw('calc_energy_og', delete_existing_job=True)
calc_energy_og_job.structure = originalStructure
calc_energy_og_job.run() # 5 min
energy_og = calc_energy_og_job.output.energy_tot[0]


The job calc_energy_og was saved and received the ID: 223


In [ ]:
# Run GPAW (Defect Structure)
calc_energy_defect_job = project.create.job.Gpaw('calc_energy_defect', delete_existing_job=True)
calc_energy_defect_job.structure = relaxedStructure
calc_energy_defect_job.run() # 13 min
energy_defect = calc_energy_defect_job.output.energy_tot[0]


The job calc_energy_defect was saved and received the ID: 225


In [ ]:
# Calculate Chemical Potential Al
chemPotential = -0.662

# Calculate Defect Formation Energy
dfe = energy_defect - energy_og + chemPotential
dfe


8.011557178731593

In [25]:
# Calculate Defect Concentration
k_B = 0.0000862
temp = 1000
concentration_defect = math.exp(-dfe/(k_B * temp))
concentration_defect

4.325337705715498e-41

In [ ]:
# calculate Al concentration
# for Al vacancy: c_Al = (1 - c_AlVac)/(2 - c_AlVac)
c_AlVac = concentration_defect
c_Al = (1 - c_AlVac)/(2 - c_AlVac)
c_Al


In [8]:
# Create Vacancy Iron
vacancyFe = originalStructure.copy()
del vacancyFe[0]
vacancyFe.plot3d()

NGLWidget()

In [9]:
# Create Antisite Iron
antisiteFe = originalStructure.copy()
antisiteFe[0] = "Al"
antisiteFe.plot3d()

NGLWidget()